#### Load data & libraries

In [14]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

In [15]:
pd.options.display.max_rows = None
pd.options.display.max_columns = None

In [17]:
# 1. Khởi tạo cấu hình kết nối SparkSession
spark = SparkSession.builder \
    .appName("Feature_Extraction") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
# 2. Đọc dữ liệu trực tiếp từ HDFS
hdfs_input_path = "hdfs://localhost:9000/big_data/hotel_bookings_cleaned.csv"
df_cleaned = spark.read.csv(hdfs_input_path, header=True, inferSchema=True)

In [5]:
df_cleaned.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: integer (nullable = true)
 |-- lead_time: integer (nullable = true)
 |-- arrival_date_year: integer (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: integer (nullable = true)
 |-- arrival_date_day_of_month: integer (nullable = true)
 |-- stays_in_weekend_nights: integer (nullable = true)
 |-- stays_in_week_nights: integer (nullable = true)
 |-- adults: integer (nullable = true)
 |-- children: integer (nullable = true)
 |-- babies: integer (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: integer (nullable = true)
 |-- previous_cancellations: integer (nullable = true)
 |-- previous_bookings_not_canceled: integer (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nulla

In [25]:
df_feature_extracted = df_cleaned \
    .withColumn("total_guests", F.col("adults") + F.col("children") + F.col("babies")) \
    .withColumn("total_nights", F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")) \
    .withColumn(
        "total_estimated_cost", 
        F.when(F.col("total_nights") == 0, F.col("adr"))
         .otherwise(F.col("adr") * F.col("total_nights"))
    ) \
    .withColumn("is_family", F.when((F.col("children") > 0) | (F.col("babies") > 0), 1).otherwise(0)) \
    .withColumn("total_past_bookings", F.col("previous_cancellations") + F.col("previous_bookings_not_canceled")) \
    .withColumn("cancellation_rate", F.col("previous_cancellations") / (F.col("total_past_bookings") + 1))

In [29]:
# Xóa các cột gây nhiễu và Target Leakage
columns_to_drop = [
    "arrival_date_year","arrival_date_week_number","arrival_date_day_of_month", "arrival_date_month",
    "assigned_room_type", "booking_changes",
    "reservation_status","reservation_status_date",
    "stays_in_weekend_nights","stays_in_week_nights",
    "adults","children","babies",
    "previous_cancellations","previous_bookings_not_canceled",
    "adr"]
df_feature_extracted = df_feature_extracted.drop(*columns_to_drop)

In [30]:
len(df_feature_extracted.columns)

21

In [31]:
df_feature_extracted.columns

['hotel',
 'is_canceled',
 'lead_time',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'reserved_room_type',
 'deposit_type',
 'agent',
 'days_in_waiting_list',
 'customer_type',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'total_guests',
 'total_nights',
 'total_estimated_cost',
 'is_family',
 'total_past_bookings',
 'cancellation_rate']

#### Data export

In [32]:
df_feature_extracted.write.csv("..\data\hotel_bookings_feature_extracted.csv", header=True, mode="overwrite")

In [33]:
spark.stop()